Findings:

1. Set resoning to False, else model takes too much time to reply.

In [ ]:
from typing import List
import time


from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.messages import AIMessage, ToolMessage
# from langchain_core.messages import ToolMessage, AIMessage


In [23]:
model_name = "qwen3.5"

reasoning_model = ChatOllama(
    model=model_name,
    validate_model_on_init=True,
    temperature=0,
    reasoning=True,
)

non_reasoning_model = ChatOllama(
    model=model_name,
    validate_model_on_init=True,
    temperature=0,
    reasoning=False,
)

In [24]:
def run_simple_prompt(llm):
    start_time=time.time()
    messages = [
        (
            "system",
            "You are a helpful assistant that translates English to French. Translate the user sentence.",
        ),
        ("human", "I love programming."),
    ]
    ai_msg = llm.invoke(messages)
    end_time = time.time()
    return ai_msg, end_time - start_time

In [25]:
run_simple_prompt(reasoning_model)

(AIMessage(content="J'aime la programmation.", additional_kwargs={'reasoning_content': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Input sentence: "I love programming."\n    *   Task: Translate from English to French.\n    *   Role: Helpful assistant.\n\n2.  **Analyze the Source Sentence:**\n    *   "I love programming."\n    *   Subject: "I" (Je)\n    *   Verb: "love" (aimer)\n    *   Object: "programming" (la programmation)\n    *   Tone: Enthusiastic, personal.\n\n3.  **Determine the Translation:**\n    *   "I" -> "J\'" (elided before a vowel) or "Je".\n    *   "love" -> "aime" (present tense, first person singular).\n    *   "programming" -> "la programmation".\n    *   Common French phrasing: "J\'aime la programmation."\n    *   Alternative (more colloquial): "J\'adore la programmation." (I adore programming) - but "love" is usually "aimer" or "adorer". "J\'aime" is the direct translation.\n    *   Another nuance: "Programming" can be "coder" (coding) or "programma

In [26]:
run_simple_prompt(non_reasoning_model)


(AIMessage(content="J'adore la programmation.", additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-03-17T05:52:06.384308592Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2553197177, 'load_duration': 126982094, 'prompt_eval_count': 37, 'prompt_eval_duration': 1625914834, 'eval_count': 7, 'eval_duration': 796539612, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019cfa59-eef5-76a3-bc58-35998cd8de12-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 7, 'total_tokens': 44}),
 2.555236339569092)

Tool Calling

In [33]:
@tool
def validate_user(user_id: int, addresses: List[str]) -> bool:
    """Validate user using historical addresses.

    Args:
        user_id (int): the user ID.
        addresses (List[str]): Previous addresses as a list of strings.
    """
    return True

In [34]:
reasoning_model_with_tool = reasoning_model.bind_tools([validate_user])

non_reasoning_model_with_tool = non_reasoning_model.bind_tools([validate_user])

In [37]:
result = non_reasoning_model_with_tool.invoke(
    "Could you validate user 123? They previously lived at "
    "123 Fake St in Boston MA and 234 Pretend Boulevard in "
    "Houston TX."
)
if isinstance(result, AIMessage) and result.tool_calls:
    print(result.tool_calls)

[{'name': 'validate_user', 'args': {'user_id': 123, 'addresses': ['123 Fake St in Boston MA', '234 Pretend Boulevard in Houston TX']}, 'id': '32d8d742-5d9d-451e-bb7d-581048ee19df', 'type': 'tool_call'}]


In [38]:
result = reasoning_model_with_tool.invoke(
    "Could you validate user 123? They previously lived at "
    "123 Fake St in Boston MA and 234 Pretend Boulevard in "
    "Houston TX."
)
if isinstance(result, AIMessage) and result.tool_calls:
    print(result.tool_calls)

[{'name': 'validate_user', 'args': {'user_id': 123, 'addresses': ['123 Fake St in Boston MA', '234 Pretend Boulevard in Houston TX']}, 'id': 'ffe8916b-7d16-4b0c-a790-de9b6f8e30c5', 'type': 'tool_call'}]


In [42]:
for call in result.tool_calls:
    print(f"Tool {calzl['name']} was called with input: {call['args']}")

Tool validate_user was called with input: {'user_id': 123, 'addresses': ['123 Fake St in Boston MA', '234 Pretend Boulevard in Houston TX']}


In [47]:
def simple_tool_call(llm):
    # Map available tools by name

    _tools_by_name = {validate_user.name: validate_user}

    # Choose which bound model to use for follow-up (non-reasoning is faster)
    _model_with_tools = llm

    # Reuse the original user prompt for context
    _user_prompt = (
        "Could you validate user 123? They previously lived at "
        "123 Fake St in Boston MA and 234 Pretend Boulevard in "
        "Houston TX."
    )

    result = _model_with_tools.invoke(_user_prompt)

    if isinstance(result, AIMessage) and result.tool_calls:
        tool_messages = []
        for call in result.tool_calls:
            tool_name = call.get("name")
            tool_args = call.get("args", {}) or {}
            tool_id = call.get("id") or call.get("tool_call_id") or ""

            tool = _tools_by_name.get(tool_name)
            if tool is None:
                tm = ToolMessage(content=f"Unknown tool: {tool_name}", tool_call_id=tool_id)
            else:
                try:
                    tool_output = tool.invoke(tool_args)
                except Exception as e:
                    tool_output = f"Tool {tool_name} failed: {e}"
                tm = ToolMessage(content=str(tool_output), tool_call_id=tool_id)
            tool_messages.append(tm)

        # Build follow-up: user -> ai(tool-call) -> tool results
        followup_messages = [("human", _user_prompt), result] + tool_messages
        final_answer = _model_with_tools.invoke(followup_messages)
        print(final_answer.content)
    else:
        print("No tool calls to resolve; result:", getattr(result, "content", result))

In [48]:
simple_tool_call(reasoning_model_with_tool)

The validation for user 123 was successful.


In [ ]:
simple_tool_call(non_reasoning_model_with_tool)